# 05 — Train the neural baselines

**What this notebook does.** Trains two from-scratch deep learning models — N-BEATS and a
DeepAR-class LSTM — on the same folds, the same horizons, and the same metrics as every
other model in the benchmark, then pushes their forecasts back to the Hub.

These are the honest comparison point for a foundation model. Chronos-2 arrives already
pretrained on millions of time series; these two start from nothing and see only our data.
The gap between them, and how that gap changes with training-set size, is the most
informative result in the project.

## Before you run this, these must already exist

| Thing | Where | Made by |
|---|---|---|
| Processed series | `rohanjain2312/forecastbench-data` → `processed/*.parquet` | build Step 15 |
| Colab Secrets | the 🔑 panel on the left | you, once |

Notebook 04 does **not** need to have finished first — these two are independent.

## The one rule this notebook follows

**No modelling logic lives here.** The fold loop, the early stopping, and the quantile
construction are all in `forecast_bench/`. Every cell imports and calls.

## Step 1 — Confirm the GPU

Runtime → Change runtime type → **H100**.

In [ ]:
!nvidia-smi

## Step 2 — Install the project

Installed from a source tarball rather than `git+https://...`: pip's git codepath shells out to the container's git binary, which has been flaky inside Colab; fetching the tarball over plain HTTP avoids that dependency entirely.

`--force-reinstall --no-deps` matters as much as the tarball switch itself: without it, pip sees `forecast-bench` already installed (the version number never changes) and silently skips reinstalling, so reopening this notebook and clicking Run All can keep running *stale* code from an earlier session if Colab reconnects you to the same live runtime rather than a fresh one. `--no-deps` keeps this fast by leaving torch, darts and chronos-forecasting alone.


In [ ]:
%pip install -q --force-reinstall --no-deps "https://github.com/Rohanjain2312/forecast_bench/archive/refs/heads/main.tar.gz"

## Step 3 — Load your credentials from Colab Secrets

Nothing is printed.

In [ ]:
import os

from google.colab import userdata

from forecast_bench.config import get_config

for key in ["HF_TOKEN", "FRED_API_KEY", "WANDB_API_KEY"]:
    try:
        os.environ[key] = userdata.get(key)
        print(f"{key}: loaded")
    except Exception:
        print(f"{key}: not set")

os.environ.setdefault("HF_DATASET_REPO", "rohanjain2312/forecastbench-data")
os.environ.setdefault("HF_MODEL_REPO", "rohanjain2312/forecastbench-chronos")

# get_config() is a process-wide singleton (functools.lru_cache) so that every
# caller in one run observes identical settings. That means if anything called
# it even once before this cell ran -- an earlier attempt, a re-run out of order --
# it is now permanently cached WITHOUT these secrets, and setting os.environ above
# would silently have no effect for the rest of this session. Clearing it here
# makes the notebook correct regardless of what was run before this cell.
get_config.cache_clear()

## Step 4 — Run the backtest with the neural models included

For a neural model, "training" and "backtesting" are the same operation: the harness
retrains the network at each block boundary and forecasts forward from every fold origin.
So this single call trains the models *and* produces their forecasts, driven by exactly the
same runner that drives ARIMA and Chronos-2.

That matters more than it sounds. The models traverse identical code, so a difference in the
results table is a difference between models rather than a difference between two
evaluation scripts that were each written on a different afternoon.

In [ ]:
from forecast_bench.backtest.runner import run_series_backtest
from forecast_bench.config import get_config, setup_logging

setup_logging("INFO")
config = get_config()
config.ensure_dirs()

spy = run_series_backtest(
    series="spy_logrv",
    cadence="matched",
    arm="A",
    include_neural=True,
    device="gpu",
)
print(f"{len(spy):,} forecast rows, models: {sorted(spy.model_id.unique())}")

## Step 5 — Save it, then do the same for the Treasury yield

In [ ]:
from forecast_bench.backtest.writer import write_results

write_results(spy, config.forecasts_dir)

dgs10 = run_series_backtest(
    series="dgs10",
    cadence="matched",
    arm="A",
    include_neural=True,
    device="gpu",
)
write_results(dgs10, config.forecasts_dir)
print(f"{len(dgs10):,} forecast rows for dgs10")

## Step 6 — The sample-efficiency sweep

The same two models trained on 1 year, 3 years, 10 years, and the full window. Plotted
against the Chronos-2 curve from notebook 04, this is the test of whether pretraining
actually buys data efficiency or just buys a head start.

`"1y"` etc. do not mean literally one year of raw observations — with context length fixed at 512 across every model in the study, a slice that short could not supply even one training example. They mean that many distinct forecast origins' worth of *additional* material; `sample_efficiency_window_size` does the conversion.


In [ ]:
import pandas as pd

from forecast_bench.models.base import (
    SAMPLE_EFFICIENCY_DAYS,
    sample_efficiency_window_size,
)

sweep = []
for name in SAMPLE_EFFICIENCY_DAYS:
    result = run_series_backtest(
        series="spy_logrv",
        cadence="matched",
        arm="A",
        include_neural=True,
        device="gpu",
        training_window_days=sample_efficiency_window_size(name),
    )
    result["training_window"] = name
    sweep.append(result)
    print(f"{name}: {len(result):,} rows")

sweep_frame = pd.concat(sweep, ignore_index=True)
sweep_frame.to_parquet(config.forecasts_dir / "spy_logrv_sample_efficiency.parquet", index=False)

## Step 7 — Push the forecasts back to the Hub

The GPU work happens here, but the scoring happens locally, through the same
`evaluation/aggregate.py` that scores every other model. Sending the forecasts back rather
than the metrics is deliberate: one scoring implementation, one set of numbers.

In [ ]:
from forecast_bench.data.hub import push_forecasts

for path in push_forecasts():
    print("uploaded", path)

## Done

**Tell Claude Code "done"** and it will pull these forecasts, run the full benchmark, and
apply the pre-registered losing condition to the result.